## Comparison between the PPP and Overton peatland policy collections

### Overton Data Acquisition

A dataset with the names of 10,000 policies was exported from Overton on July 12th 2026 using the following criteria:
Search string: Peatland OR Peat OR Wetland
Search results filtered by:
- Source sector – Public sector
-	Source Organisation type – Government, Legislative Body
-	Years – 2000 to 2026
-	Source Country: Every EU country and the UK
-	Document type: Publication
Sorted by: Relevance

This search and subsequent filtering returned 34,872 policies, of which the top 10,000 most relevant were exported into "Overton data NEW.csv".



In [111]:
!! pip install pandas thefuzz tqdm

['WARNING: Ignoring invalid distribution -ip (c:\\python310\\lib\\site-packages)',
 'WARNING: Ignoring invalid distribution -atplotlib (c:\\python310\\lib\\site-packages)',
 'WARNING: Ignoring invalid distribution -ip (c:\\python310\\lib\\site-packages)',
 'WARNING: Ignoring invalid distribution -atplotlib (c:\\python310\\lib\\site-packages)',
 'Requirement already satisfied: pandas in c:\\python310\\lib\\site-packages (2.3.2)',
 'Requirement already satisfied: thefuzz in c:\\python310\\lib\\site-packages (0.22.1)',
 'Requirement already satisfied: tqdm in c:\\python310\\lib\\site-packages (4.67.0)',
 'Requirement already satisfied: pytz>=2020.1 in c:\\python310\\lib\\site-packages (from pandas) (2024.2)',
 'Requirement already satisfied: numpy>=1.22.4 in c:\\python310\\lib\\site-packages (from pandas) (2.2.6)',
 'Requirement already satisfied: python-dateutil>=2.8.2 in c:\\python310\\lib\\site-packages (from pandas) (2.9.0.post0)',
 'Requirement already satisfied: tzdata>=2022.7 in c:

In [ ]:
import pandas as pd
from thefuzz import process as fuzzy_process
from tqdm import tqdm
import re
import os
import glob
import numpy as np

cwd = os.getcwd()

# --- V9 FINAL CONFIGURATION ---
ODOO_NEW_PATH = f'{cwd}/Data/OdooData_1960_2026.csv'
OVERTON_PATHS = glob.glob(f'{cwd}/Data/OvertonData_*.csv')

FUZZY_MATCH_THRESHOLD = 95

FINAL_OUTPUT_PATH_SUMMARY = f'{cwd}/Outputs/ppp_overton_level_summary_thresh{FUZZY_MATCH_THRESHOLD}.csv'
FINAL_OUTPUT_PATH_DETAILS = f'{cwd}/Outputs/ppp_overton_level_details_thresh{FUZZY_MATCH_THRESHOLD}.csv'

# List of primary countries for consolidation.
# This ensures we correctly group subdivisions.
EU27_UK_COUNTRIES = [
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic',
    'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary',
    'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta',
    'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia',
    'Spain', 'Sweden', 'United Kingdom'
]

def consolidate_overton_country(country_name: str) -> str:
    """
    Consolidates Overton's subdivided country names into a single parent country.
    Example: 'Germany: Bavaria' -> 'Germany'
    """
    if not isinstance(country_name, str):
        return 'Unknown'
    # Special handling for UK, as it's often used as a prefix.
    if country_name.startswith('UK'):
        return 'United Kingdom'
    for parent_country in EU27_UK_COUNTRIES:
        if country_name.startswith(parent_country):
            return parent_country
    # If it's not a subdivision of our target countries, return it as is (e.g., 'EU').
    return country_name

def attempt_reciprocal_match(
    ppp_search_title: str,
    candidate_overton_titles: list,
    candidate_ppp_titles: list,
    threshold: int
) -> dict:
    """
    Performs a high-confidence reciprocal match.
    """
    if not isinstance(ppp_search_title, str) or not ppp_search_title.strip() or not candidate_overton_titles:
        return None

    forward_match = fuzzy_process.extractOne(ppp_search_title, candidate_overton_titles)

    if forward_match and forward_match[1] >= threshold:
        best_overton_title, score = forward_match
        if candidate_ppp_titles:
            reverse_match = fuzzy_process.extractOne(best_overton_title, candidate_ppp_titles)
            if reverse_match and reverse_match[0] == ppp_search_title:
                return {'overton_title': best_overton_title, 'similarity_score': score}
    return None

Main analysis script to determine the coverage of different policy levels in Overton.

In [113]:
ppp_df = pd.read_csv(ODOO_NEW_PATH, usecols=["ID","Name","Name (Native Language)","Level","Country"], engine='python')
ppp_df.rename(columns={'Name': 'title_en', 'Name (Native Language)': 'title_native'}, inplace=True)

# Fill NaN countries with a placeholder for grouping.
ppp_df['Country'].fillna('No Country', inplace=True)

print(len(ppp_df))

571


C:\Users\Ales\AppData\Local\Temp\ipykernel_32488\1291933902.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ppp_df['Country'].fillna('No Country', inplace=True)


In [ ]:
# have to merge dfs because overton exports MAX 10000 results at a time
# so broke up the search into different year spans at roughly 8000 each
df_lst = []
for ovdf_addr in OVERTON_PATHS:
    ovdf = pd.read_csv(ovdf_addr, usecols=["Overton id", 'Title', "Translated title", 'Source country'], engine='python', on_bad_lines='warn')
    df_lst.append(ovdf)

overton_df = pd.concat(df_lst)
overton_df.rename(columns={"Overton id":"ID", 'Title': 'overton_title', "Translated title":"translated_title", 'Source country': 'overton_country'}, inplace=True)

# Apply the consolidation logic to the Overton country data.
overton_df['consolidated_country'] = overton_df['overton_country'].apply(consolidate_overton_country)
overton_titles_by_country = overton_df.groupby('consolidated_country')['overton_title'].apply(list).to_dict()
print(f"Processed {len(overton_df)} policies from Overton and consolidated country names.")


C:\Users\Ales\AppData\Local\Temp\ipykernel_32488\2467946706.py:3: ParserWarning: Skipping line 3063: ',' expected after '"'

  ovdf = pd.read_csv(ovdf_addr, usecols=["Overton id", 'Title', "Translated title", 'Source country'], engine='python', on_bad_lines='warn')
C:\Users\Ales\AppData\Local\Temp\ipykernel_32488\2467946706.py:3: ParserWarning: Skipping line 5731: ',' expected after '"'

  ovdf = pd.read_csv(ovdf_addr, usecols=["Overton id", 'Title', "Translated title", 'Source country'], engine='python', on_bad_lines='warn')
C:\Users\Ales\AppData\Local\Temp\ipykernel_32488\2467946706.py:3: ParserWarning: Skipping line 8828: ',' expected after '"'

  ovdf = pd.read_csv(ovdf_addr, usecols=["Overton id", 'Title', "Translated title", 'Source country'], engine='python', on_bad_lines='warn')
C:\Users\Ales\AppData\Local\Temp\ipykernel_32488\2467946706.py:3: ParserWarning: Skipping line 5989: ',' expected after '"'

  ovdf = pd.read_csv(ovdf_addr, usecols=["Overton id", 'Title', "Translated t

Processed 39793 policies from Overton and consolidated country names.


#### Skipped Overton Entries

1959to2014:L3063 [ministrukabinets-a12757765f56ae4efbcadcfedc863bc1]: "\"Tuned nature management in transboundary area of Estonia and Latvia\""" (\"""Green Corridor\""")""" [Latvia] "The document describes the Estonia-Latvia Programme 2007-2013 project 'Tuned nature management in transboundary area of Estonia and Latvia' also known as 'Green Corridor'."

1959to2014:L5731 [govdeptsfrance-85a871b6342101fddbf324b881c5296e]: "22 fiches - lauréats appel à projets « innovation et partenariat »","22 files - winners of the call for &quot;innovation and partnership&quot; projects" [France] "The document presents a list of research projects funded by the CASDAR program, which aims to support innovation and partnership in agriculture."

1959to2014:L8828 [stateofsaxonyanhalt-21ec59bc3fb7b359d43fe3ef493bd934]: "Water Development Concept \"Mulde/Biese\"" [Germany] "The document is a Gewässerentwicklungskonzept (water development concept) for the Milde/Biese region in Sachsen-Anhalt, Germany. It aims to provide a comprehensive overview of the region's water bodies and to develop a plan for their sustainable development."

2015to2018:L5989 [govdeptsfrance-a3d1bc1f0dc8f9a6874b021616a1cafe]: "AAP RFI 2013 n° 1331…" [France] "The document presents a list of 16 research projects funded by the CASDAR program, which focuses on developing innovative technologies for agriculture."

2015to2018:L6087 [govdeptsfrance-68b1f28d7505344ed17f1e1b7ef73782]: "2018 Casdar" [France] "The document presents the results of the 2018 CASDAR project, which aimed to develop and evaluate tools for agricultural research and development."

2015to2018:L6310 [govdeptsfrance-7aa21b6b9f160d4af2e29537ef970f84]:"AAP RFI 2013 n° 1331…" [France] "The document presents 18 projects funded by the DAR, which are focused on various aspects of agriculture and food production in France."

2019to2022:L5583 [govdeptsfrance-97f14dcf610344cade44df448b527660]: "Programme national de développement agricole et rural" [France] "The document discusses various agricultural projects in France, including the development of a new system for raising chickens, the optimization of cow milking, and the production of almonds in organic farming."

In [115]:
# 3. Perform Matching
# [takes about 2 minutes]
results = []
print("\nBeginning cross-referencing...")
for _, ppp_row in tqdm(ppp_df.iterrows(), total=len(ppp_df), desc="Matching Policies"):
    ppp_country = ppp_row['Country']
    match_found = False

    # Define the search spaces for this PPP policy
    search_countries = []
    if ppp_country in EU27_UK_COUNTRIES:
        search_countries.append(ppp_country)
    search_countries.append('EU') # Always search the 'EU' bucket as requested

    # Get all possible PPP titles for reciprocal check
    candidate_ppp_titles = [t for t in [ppp_row['title_en'], ppp_row['title_native']] if pd.notna(t)]

    for search_country in search_countries:
        candidate_overton_titles = overton_titles_by_country.get(search_country, [])
        if not candidate_overton_titles:
            continue

        for ppp_title in candidate_ppp_titles:
            match_info = attempt_reciprocal_match(ppp_title, candidate_overton_titles, candidate_ppp_titles, FUZZY_MATCH_THRESHOLD)
            if match_info:
                match_found = True
                break
        if match_found:
            break
    
    results.append({
        'ppp_country': ppp_country,
        'ppp_level': ppp_row['Level'],
        'match_status': 'Match' if match_found else 'Not in Overton'
    })

results_df = pd.DataFrame(results)


Beginning cross-referencing...


Matching Policies: 100%|██████████| 571/571 [01:50<00:00,  5.15it/s]


In [116]:
results_df

,ppp_country,ppp_level,match_status
0,No Country,Global,Not in Overton
1,No Country,Global,Not in Overton
2,No Country,Global,Not in Overton
3,No Country,Global,Not in Overton
4,No Country,Global,Not in Overton
...,...,...,...
566,France,National,Not in Overton
567,France,National,Not in Overton
568,France,National,Not in Overton
569,Malta,National,Not in Overton


Generate and save output tables

In [117]:
# Table 1: Summary of matches by policy level (unchanged)
level_summary = results_df.groupby('ppp_level')['match_status'].value_counts().unstack(fill_value=0)
level_summary['Total from PPP'] = level_summary.sum(axis=1)
level_summary['Match_Percentage'] = (level_summary['Match'] / level_summary['Total from PPP'] * 100).round(2)

print("\n--- Table 1: Overton Match Rate by Policy Level ---")
print(level_summary.to_string())
level_summary.to_csv(FINAL_OUTPUT_PATH_SUMMARY)
print(f"Summary table saved to: {FINAL_OUTPUT_PATH_SUMMARY}")


--- Table 1: Overton Match Rate by Policy Level ---
match_status  Match  Not in Overton  Total from PPP  Match_Percentage
ppp_level                                                            
European          0              21              21              0.00
Global            0              10              10              0.00
Local             1              32              33              3.03
National         53             344             397             13.35
Regional         12              98             110             10.91
Summary table saved to: c:\Users\Ales\Documents\GitHub\PPP_v_Overton_Data_Analysis/Outputs/ppp_overton_level_summary_thresh95.csv


In [118]:
overton_reg_ct = {}
for region in set(list(overton_df['overton_country'])):
    if region=="UK":
        overton_reg_ct["United Kingdom"]=len(overton_df[overton_df['overton_country']==region])
    overton_reg_ct[region]=len(overton_df[overton_df['overton_country']==region])
overton_reg_ct

{'Germany': 1019,
 'Hungary': 118,
 'Netherlands': 656,
 'Slovakia': 179,
 'France': 325,
 'Ireland': 4373,
 'Portugal': 157,
 'Finland': 1062,
 'Lithuania': 10,
 'Romania': 219,
 'Malta': 5,
 'Cyprus': 133,
 'Denmark': 243,
 'Sweden': 2120,
 'Belgium': 440,
 'Croatia': 109,
 'Poland': 100,
 'Spain': 1205,
 'Bulgaria': 341,
 'Slovenia': 92,
 'EU': 7618,
 'Czech Republic': 72,
 'Greece': 137,
 'Austria': 645,
 'Luxembourg': 12,
 'Estonia': 602,
 'Italy': 227,
 'United Kingdom': 16944,
 'UK': 16944,
 'Latvia': 630}

In [119]:
# Table 2: Updated to be a summary by country only
country_summary = results_df.groupby('ppp_country')['match_status'].value_counts().unstack(fill_value=0)
# Ensure 'Match' and 'No Match' columns exist to prevent errors if one is missing
if 'Match' not in country_summary:
    country_summary['Match'] = 0
if 'Not in Overton' not in country_summary:
    country_summary['Not in Overton'] = 0

country_summary['Total from PPP'] = country_summary['Match'] + country_summary['Not in Overton']
country_summary['Match_Percentage'] = (country_summary['Match'] / country_summary['Total from PPP'] * 100).round(2)

print("\n--- Table 2: Detailed Breakdown by Country ---")
# We can drop the 'No Country' category from the final display table for clarity
country_summary = country_summary.drop('No Country', errors='ignore')

country_summary["Total from Overton"] = np.nan
for country in country_summary.index:
    if "Country" not in country:
        country_summary.loc[country, "Total from Overton"] = overton_reg_ct[country]

print(country_summary.to_string())

country_summary.to_csv(FINAL_OUTPUT_PATH_DETAILS)
print(f"Detailed breakdown saved to: {FINAL_OUTPUT_PATH_DETAILS}")


--- Table 2: Detailed Breakdown by Country ---
match_status    Match  Not in Overton  Total from PPP  Match_Percentage  Total from Overton
ppp_country                                                                                
Austria             2               7               9             22.22               645.0
Belgium             0              25              25              0.00               440.0
Bulgaria            4               9              13             30.77               341.0
Croatia             1               8               9             11.11               109.0
Cyprus              0              16              16              0.00               133.0
Czech Republic      1              13              14              7.14                72.0
Denmark             1              10              11              9.09               243.0
Estonia             4              20              24             16.67               602.0
Finland             1           

#### Qualitative Analysus: Luxembourg

In [120]:
# overton titles in original language
overton_df[overton_df["overton_country"]=="Luxembourg"]["overton_title"]

5148                                    Dossier consolidé
5728                       Bulletin spécial novembre 1980
7262    Renaturation des cours d'eau - Restauration de...
7636                 Bulletin de documentation n° 15/1965
8758                    DEBAT D’ORIENTATION sur la chasse
7213                                        Projet De Loi
8299        Journal officiel du Grand-Duché de Luxembourg
9225                                        Projet De Loi
8579    Rapport d'activité 2020 du Musée national d’hi...
5080                                      Ordre du jour :
7033                                      Nom du document
7034                                      Nom du document
Name: overton_title, dtype: object

In [121]:
# overton titles in english
overton_df[overton_df["overton_country"]=="Luxembourg"]["translated_title"]

5148                                    Consolidated file
5728                       Special Bulletin November 1980
7262    Renaturation of waterways - Restoration of wet...
7636                   Documentation Bulletin No. 15/1965
8758                      DEBATE ON THE POLICY OF HUNTING
7213                                                 Bill
8299    Official Journal of the Grand Duchy of Luxembourg
9225                                                 Bill
8579    2020 activity report of the National Museum of...
5080                                             Agenda :
7033                                        Document name
7034                                        Document name
Name: translated_title, dtype: object

In [122]:
# portal titles in original language
ppp_df[ppp_df['Country']=="Luxembourg"]['title_native']

281    Plan National concernant la Protection de la N...
282    Plan Stratégique National du Grand-Duché de Lu...
283    Plan national intégré en matière d’énergie et ...
284    Projet de Stratégie et plan d’action pour l’ad...
299                    Plan comptable forestier national
330    Luxembourg Klimabonus Bësch Mouer a Wiss Subve...
491    Document d'information relatif au 3e Plan Nati...
499    Plan National concernant la Protection de la N...
531    Dritter Bewirtschaftungsplan für die luxemburg...
Name: title_native, dtype: object

In [123]:
# portal titles in english
ppp_df[ppp_df['Country']=="Luxembourg"]['title_en']

281    National Nature Conservation Plan - 3rd Plan t...
282    CAP - National Strategic Plan of the Grand-Duc...
283    Integrated National Plan on energy and climate...
284    Draft strategy and action plan for adaptation ...
299         National Forestry Accounting Plan Luxembourg
330    Luxembourg Climate Bonus Forest Moor and Meado...
491    Information document relating to the 3rd Natio...
499           National Plan for the Protection of Nature
531    Third Management Plan For The Luxembourg Porti...
Name: title_en, dtype: object